In [23]:
# Import Necessary Packages/Modules
import time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from skeLCS import eLCS   # pip install scikit-eLCS
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix

#Reusable constants.  You can change these to match your own data and preferences.
DATA_FILE  = "medical_insurance_cleaned_v2.csv"   # output of Task 3
SEED       = 42
N_SPLITS   = 3      # same 3-fold CV as the baseline
N_REPEATS  = 1      # raise to 5 for a meaningful significance test (15 scores per model, much slower)
N_FEATURES = 4      # features kept by the selector
N_JOBS     = -1     # folds run in parallel; set to 1 if you hit memory/pickling problems

In [24]:
# Define columns that may introduce data leakage
#These values are derived from the target variable and should not be used as features in the model to avoid data leakage
LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost",
                "claims_count", "avg_claim_amount", "total_claims_paid"]

#converting the risk_score into 3 bands (low, medium, high) using quantiles and storing it in a new column called risk_band
df = pd.read_csv(DATA_FILE, keep_default_na=False, na_values=[""])   # keeps "None" in alcohol_freq as a category
y = pd.qcut(df["risk_score"], 3, labels=False).values                # 0 = low, 1 = medium, 2 = high
BAND_NAMES = {0: "Low", 1: "Medium", 2: "High"}

# Prepare the feature matrix X by dropping the leakage columns, risk_score, and person_id. Convert categorical columns to integer codes for model compatibility.
X_df = df.drop(columns=LEAKAGE_COLS + ["risk_score", "person_id"])
for c in X_df.select_dtypes(exclude="number"):
    X_df[c] = X_df[c].astype("category").cat.codes                   # text -> integer codes
feature_names = list(X_df.columns)
X = X_df.values.astype(float)
print("X:", X.shape, "| class shares:", np.round(np.bincount(y) / len(y), 3))

X: (96713, 45) | class shares: [0.349 0.328 0.322]


In [25]:
# Define a feature selector function that uses a shallow decision tree to select the N_FEATURES most important features. This function will be used in the pipeline to ensure that feature selection is performed on each training fold without leaking information from the test fold.
def selector():
    # keeps the N_FEATURES most important columns according to a shallow tree.
    # Inside a Pipeline it is refit on each training fold only, so the test fold never influences selection.
    return SelectFromModel(DecisionTreeClassifier(max_depth=5, random_state=SEED),
                           max_features=N_FEATURES, threshold=-np.inf)
class ScikitELCSWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, N=1000, learning_iterations=10000, do_correct_set_subsumption=False, random_state=None):
        self.N = N
        self.learning_iterations = learning_iterations
        self.do_correct_set_subsumption = do_correct_set_subsumption
        self.random_state = random_state

    def fit(self, X, y):
        y_int = y.astype(int)
        self.model_ = eLCS(
            N=self.N,
            learning_iterations=self.learning_iterations,
            do_correct_set_subsumption=self.do_correct_set_subsumption,
            random_state=self.random_state
        )
        self.model_.fit(X, y_int)
        self.classes_ = np.unique(y_int)
        return self

    def predict(self, X):
        if hasattr(self.model_, "predict"):
            return self.model_.predict(X)
        elif hasattr(self.model_, "predict_label"):
            return self.model_.predict_label(X)
        else:
            raise AttributeError("eLCS instance has no valid prediction method.")

configs = {
    "A. Original eLCS, preprocessed data (45 features)": Pipeline([
        ("elcs", ScikitELCSWrapper(random_state=SEED))
    ]),
    "B. + feature selection": Pipeline([
        ("select", selector()),
        ("elcs", ScikitELCSWrapper(random_state=SEED))
    ]),
    "C. + MinMax scaling": Pipeline([
        ("select", selector()), 
        ("scale", MinMaxScaler()),
        ("elcs", ScikitELCSWrapper(random_state=SEED))
    ]),
    "D. + correct-set subsumption": Pipeline([
        ("select", selector()), 
        ("scale", MinMaxScaler()),
        ("elcs", ScikitELCSWrapper(random_state=SEED, do_correct_set_subsumption=True))
    ]),
    "E. Improved (+ 20k iterations, N=1500)": Pipeline([
        ("select", selector()), 
        ("scale", MinMaxScaler()),
        ("elcs", ScikitELCSWrapper(
            random_state=SEED, 
            do_correct_set_subsumption=True,
            learning_iterations=20000, 
            N=1500
        ))
    ]),
}
IMPROVED = "E. Improved (+ 20k iterations, N=1500)"
BASELINE = "A. Original eLCS, preprocessed data (45 features)"

In [26]:
# Run cross-validation for each configuration and collect scores and execution times. The scoring metrics used are balanced accuracy and macro F1 score. The results are printed for each configuration, showing the mean and standard deviation of the balanced accuracy, the mean macro F1 score, and the time taken for cross-validation.
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
scoring = {"bal_acc": "balanced_accuracy", "f1_macro": "f1_macro"}  #scoring metrics to be used for cross-validation

scores, times = {}, {} #containers to hold the cross-validation scores and execution times for each configuration
for name, pipe in configs.items():
    t0 = time.time()
    res = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=N_JOBS)
    scores[name] = res
    times[name] = time.time() - t0
    print(f"{name}\n   balanced acc {res['test_bal_acc'].mean():.3f} +/- {res['test_bal_acc'].std():.3f}"
          f" | macro-F1 {res['test_f1_macro'].mean():.3f} | {times[name]:.0f}s")

A. Original eLCS, preprocessed data (45 features)
   balanced acc 0.542 +/- 0.021 | macro-F1 0.511 | 44s
B. + feature selection
   balanced acc 0.787 +/- 0.034 | macro-F1 0.767 | 35s
C. + MinMax scaling
   balanced acc 0.787 +/- 0.034 | macro-F1 0.767 | 33s
D. + correct-set subsumption
   balanced acc 0.828 +/- 0.028 | macro-F1 0.818 | 29s
E. Improved (+ 20k iterations, N=1500)
   balanced acc 0.866 +/- 0.011 | macro-F1 0.863 | 44s


In [27]:
summary = pd.DataFrame({
    "Balanced accuracy (mean)": {n: r["test_bal_acc"].mean() for n, r in scores.items()},
    "Std across folds":         {n: r["test_bal_acc"].std()  for n, r in scores.items()},
    "Macro-F1 (mean)":          {n: r["test_f1_macro"].mean() for n, r in scores.items()},
    "Time (s)":                 times,
}).round(3)
summary

,Balanced accuracy (mean),Std across folds,Macro-F1 (mean),Time (s)
"A. Original eLCS, preprocessed data (45 features)",0.542,0.021,0.511,43.897
B. + feature selection,0.787,0.034,0.767,34.630
C. + MinMax scaling,0.787,0.034,0.767,33.434
D. + correct-set subsumption,0.828,0.028,0.818,28.514
"E. Improved (+ 20k iterations, N=1500)",0.866,0.011,0.863,43.879


In [28]:
#Compare scores with the Nadeau-Bengio test
def nadeau_bengio(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b)                           #difference between the two sets of scores
    k = len(d)                                                  #number of folds (or number of paired scores)
    var = d.var(ddof=1)                                         #variance of the differences
    if var == 0:                                                #if the variance is zero, return NaN for both t-statistic and p-value
        return np.nan, np.nan                                   #return NaN for both t-statistic and p-value
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * var)
    return t, 2 * stats.t.sf(abs(t), k - 1)

n_test  = int(len(y) / N_SPLITS)
n_train = len(y) - n_test
base = scores[BASELINE]["test_bal_acc"]

rows = []
for name, r in scores.items():
    if name == BASELINE:
        continue
    s = r["test_bal_acc"]
    try:
        w_p = stats.wilcoxon(s, base).pvalue
    except ValueError:                     # all differences zero
        w_p = np.nan
    t, nb_p = nadeau_bengio(s, base, n_train, n_test)
    rows.append({"Configuration vs baseline": name, "Mean diff": s.mean() - base.mean(),
                 "Wilcoxon p": w_p, "Nadeau-Bengio p": nb_p})
pd.DataFrame(rows).round(4)

,Configuration vs baseline,Mean diff,Wilcoxon p,Nadeau-Bengio p
0,B. + feature selection,0.2444,0.25,0.0440
1,C. + MinMax scaling,0.2444,0.25,0.0440
2,D. + correct-set subsumption,0.2854,0.25,0.0352
3,"E. Improved (+ 20k iterations, N=1500)",0.3241,0.25,0.0059


In [ ]:
# Hold-out evaluation of the final model on a separate test set. 
from sklearn.metrics import classification_report
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

final = clone(configs[IMPROVED]).fit(Xtr, ytr)
elcs  = final.named_steps["elcs"]
pred  = final.predict(Xte)

print(f"Hold-out balanced accuracy: {balanced_accuracy_score(yte, pred):.3f}")
print(f"Hold-out macro-F1:          {f1_score(yte, pred, average='macro'):.3f}")

# Point directly to the internal model_ attribute:
print(f"Training accuracy of final rule set: {elcs.model_.get_final_accuracy():.3f}")
print(f"Training instance coverage:          {elcs.model_.get_final_instance_coverage():.1%}")

print("\nConfusion matrix (rows = true, cols = predicted):\n", confusion_matrix(yte, pred))
print("\n", classification_report(yte, pred, target_names=[BAND_NAMES[i] for i in range(3)], digits=3))
print("Selected features:", [f for f, keep in zip(feature_names, final.named_steps["select"].get_support()) if keep])

Hold-out balanced accuracy: 0.854
Hold-out macro-F1:          0.848


AttributeError: 'ScikitELCSWrapper' object has no attribute 'get_final_accuracy'

In [ ]:
# Pruning function to filter rules based on accuracy and numerosity
def prune(model, min_acc=0.9, min_num=2):
    return [r for r in model.population.pop_set if r.accuracy >= min_acc and r.numerosity >= min_num]

full_pop = elcs.population.pop_set
print("Rules before compaction (macro):", len(full_pop),
      "| micro (sum of numerosity):", sum(r.numerosity for r in full_pop))

for min_acc, min_num in [(0.0, 1), (0.8, 2), (0.9, 2), (0.9, 5)]:
    kept = prune(elcs, min_acc, min_num)
    elcs.population.pop_set = kept
    p = final.predict(Xte)
    print(f"min_acc={min_acc}, min_num={min_num}: {len(kept):4d} rules | "
          f"balanced acc {balanced_accuracy_score(yte, p):.3f}")
elcs.population.pop_set = full_pop           # restore the full rule set

Baseline eLCS (45 features), CV balanced accuracy: 0.568
Fold scores: [0.505 0.641 0.56 ]
CPU times: total: 2min 59s
Wall time: 3min 30s


In [ ]:
selected = [f for f, keep in zip(feature_names, final.named_steps["select"].get_support()) if keep]
scaler = final.named_steps["scale"]

def describe(rule):
    parts = []
    for idx, cond in zip(rule.specified_attributes, rule.condition):
        lo_s, rng = scaler.data_min_[idx], scaler.data_range_[idx]
        if isinstance(cond, (list, tuple, np.ndarray)):
            parts.append(f"{selected[idx]} in [{cond[0]*rng + lo_s:.2f}, {cond[1]*rng + lo_s:.2f}]")
        else:
            parts.append(f"{selected[idx]} = {cond*rng + lo_s:.2f}")
    return " AND ".join(parts)

top = sorted(elcs.population.pop_set, key=lambda r: r.numerosity * r.fitness, reverse=True)[:10]
rule_table = pd.DataFrame([{
    "IF": describe(r), "THEN": BAND_NAMES[int(r.phenotype)],
    "accuracy": round(r.accuracy, 3), "fitness": round(r.fitness, 3), "numerosity": r.numerosity
} for r in top])
pd.set_option("display.max_colwidth", 120)
rule_table

Training accuracy: 0.488
Instance coverage: 74.5%
CPU times: total: 3min 41s
Wall time: 4min 41s
